In [1]:
import os
import requests
import json
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler

import re

In [2]:
test_2024_raw = pd.read_csv("test_2024.csv")
train_2023_raw = pd.read_csv("train_2023.csv")
train_2022_raw = pd.read_csv("train_2022.csv")
train_2021_raw = pd.read_csv("train_2021.csv")

In [3]:
train_raw_df = pd.concat([train_2021_raw, train_2022_raw, train_2023_raw])

In [51]:
train_raw_df["Unnamed: 4_level_0_Squad"].nunique()

120

In [53]:
train_raw_df.info()

<class 'pandas.DataFrame'>
Index: 9009 entries, 0 to 2965
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0                   9009 non-null   int64  
 1   Unnamed: 0_x                 9009 non-null   int64  
 2   Unnamed: 0_level_0_Rk        9009 non-null   str    
 3   Name_x                       9009 non-null   str    
 4   Unnamed: 2_level_0_Nation    9007 non-null   str    
 5   Unnamed: 3_level_0_Pos       9009 non-null   str    
 6   Unnamed: 4_level_0_Squad     9009 non-null   str    
 7   Unnamed: 5_level_0_Age       9009 non-null   str    
 8   Unnamed: 6_level_0_Born      9009 non-null   str    
 9   Playing Time_MP              9009 non-null   str    
 10  Playing Time_Starts          9009 non-null   str    
 11  Playing Time_Min             9009 non-null   str    
 12  Playing Time_90s             9009 non-null   str    
 13  Performance_Gls              9009 

In [5]:
train_raw_df.head(30)

,Unnamed: 0,Unnamed: 0_x,Unnamed: 0_level_0_Rk,Name_x,Unnamed: 2_level_0_Nation,Unnamed: 3_level_0_Pos,Unnamed: 4_level_0_Squad,Unnamed: 5_level_0_Age,Unnamed: 6_level_0_Born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,Unnamed: 24_level_0_Matches,Name_Clean,Unnamed: 0_y,Name_y,Value (€)
0,0,0,1,Max Aarons,eng ENG,DF,Norwich City,21,2000,34,32,2881,32.0,0,2,2,0,0,0,8,0,0.00,0.06,0.06,0.00,0.06,Matches,max aarons,NaN,NaN,NaN
1,1,1,2,Che Adams,sct SCO,FW,Southampton,25,1996,30,23,2039,22.7,7,3,10,7,0,0,0,0,0.31,0.13,0.44,0.31,0.44,Matches,che adams,NaN,NaN,NaN
2,2,2,3,Rayan Aït-Nouri,dz ALG,MF,Wolves,20,2001,23,20,1828,20.3,1,2,3,1,0,0,4,0,0.05,0.10,0.15,0.05,0.15,Matches,rayan ait-nouri,673.0,Rayan Aït-Nouri,22000000.0
3,3,3,4,Kristoffer Ajer,no NOR,DF,Brentford,23,1998,24,23,1995,22.2,1,3,4,1,0,0,5,0,0.05,0.14,0.18,0.05,0.18,Matches,kristoffer ajer,453.0,Kristoffer Ajer,16000000.0
4,4,4,5,Nathan Aké,nl NED,DF,Manchester City,26,1995,14,10,923,10.3,2,0,2,2,0,0,0,0,0.20,0.00,0.20,0.20,0.20,Matches,nathan ake,5.0,Nathan Aké,42000000.0
5,5,5,6,Marc Albrighton,eng ENG,"MF,FW",Leicester City,31,1989,17,11,1132,12.6,1,2,3,1,0,0,3,0,0.08,0.16,0.24,0.08,0.24,Matches,marc albrighton,NaN,NaN,NaN
6,6,6,7,Thiago Alcántara,es ESP,MF,Liverpool,30,1991,25,17,1534,17.0,1,4,5,1,0,0,2,0,0.06,0.23,0.29,0.06,0.29,Matches,thiago alcantara,129.0,Thiago Alcántara,15000000.0
7,7,7,8,Trent Alexander-Arnold,eng ENG,DF,Liverpool,22,1998,32,32,2853,31.7,2,12,14,2,0,0,2,0,0.06,0.38,0.44,0.06,0.44,Matches,trent alexander-arnold,123.0,Trent Alexander-Arnold,65000000.0
8,8,8,9,Alisson,br BRA,GK,Liverpool,28,1992,36,36,3240,36.0,0,1,1,0,0,0,0,0,0.00,0.03,0.03,0.00,0.03,Matches,alisson,108.0,Alisson,35000000.0
9,9,9,10,Allan,br BRA,MF,Everton,30,1991,28,25,2183,24.3,0,2,2,0,0,0,7,1,0.00,0.08,0.08,0.00,0.08,Matches,allan,501.0,Allan,6000000.0


In [6]:
squad_rank_map = {
    # [21-22, 22-23, 23-24, 24-25]
    'Ajaccio': [22, 18, 22, 22], 'Alavés': [20, 22, 10, 12], 'Almería': [22, 17, 19, 22], 'Angers': [14, 20, 22, 18],
    'Arminia': [17, 22, 22, 22], 'Arsenal': [5, 2, 2, 2], 'Aston Villa': [14, 7, 4, 5], 'Atalanta': [8, 5, 4, 3],
    'Athletic Club': [8, 8, 5, 6], 'Atlético Madrid': [3, 3, 4, 3], 'Augsburg': [14, 15, 11, 13], 'Auxerre': [22, 17, 22, 15],
    'Barcelona': [2, 1, 2, 1], 'Bayern Munich': [1, 1, 3, 1], 'Bochum': [13, 14, 16, 17], 'Bologna': [13, 9, 5, 8],
    'Bordeaux': [20, 22, 22, 22],'Bournemouth': [22, 15, 12, 11],'Brentford': [13, 9, 16, 14],'Brest': [11, 14, 3, 9],
    'Brighton': [9, 6, 11, 10],'Burnley': [18, 22, 19, 22],'Cagliari': [18, 22, 16, 16],'Celta Vigo': [11, 13, 14, 11],
    'Chelsea': [3, 12, 6, 4],'Clermont Foot': [17, 8, 18, 22],'Cremonese': [22, 19, 22, 22],'Crystal Palace': [12, 11, 10, 12],
    'Cádiz': [17, 14, 18, 22],'Darmstadt 98': [22, 22, 18, 22],'Dortmund': [2, 2, 5, 4],'Eintracht Frankfurt': [11, 7, 6, 6],
    'Elche': [13, 20, 22, 22],'Empoli': [14, 14, 17, 14],'Espanyol': [14, 19, 22, 17],'Everton': [16, 17, 15, 15],
    'Fiorentina': [7, 8, 8, 7], 'Freiburg': [6, 5, 10, 8], 'Frosinone': [22, 22, 18, 22],
    'Fulham': [22, 10, 13, 12], 'Genoa': [19, 22, 11, 13], 'Getafe': [15, 15, 12, 14], 'Girona': [22, 10, 3, 7],
    'Gladbach': [10, 10, 14, 11], 'Granada': [18, 22, 20, 22], 'Greuther Fürth': [18, 22, 22, 22], 'Heidenheim': [22, 22, 8, 12],
    'Hellas Verona': [9, 18, 13, 15], 'Hertha BSC': [16, 18, 22, 22], 'Hoffenheim': [9, 12, 7, 10], 'Inter': [2, 3, 1, 2],
    'Juventus': [4, 7, 3, 3], 'Köln': [7, 11, 17, 22], 'Las Palmas': [22, 22, 16, 17], 'Lazio': [5, 2, 7, 6],
    'Le Havre': [22, 22, 15, 16], 'Lecce': [22, 16, 14, 15], 'Leeds United': [17, 19, 22, 22], 'Leicester City': [8, 18, 22, 16],
    'Lens': [7, 2, 7, 8], 'Levante': [19, 22, 22, 22], 'Leverkusen': [3, 6, 1, 3], 'Lille': [10, 5, 4, 5],
    'Liverpool': [2, 5, 3, 1], 'Lorient': [16, 10, 19, 22], 'Luton Town': [22, 22, 18, 22], 'Lyon': [8, 7, 6, 6],
    'Mainz 05': [8, 9, 13, 12], 'Mallorca': [16, 9, 15, 13], 'Manchester City': [1, 1, 1, 2], 'Manchester Utd': [6, 3, 8, 7],
    'Marseille': [2, 3, 8, 5], 'Metz': [19, 22, 18, 22], 'Milan': [1, 4, 2, 4], 'Monaco': [3, 6, 2, 2],
    'Montpellier': [13, 12, 12, 14], 'Monza': [22, 11, 12, 15], 'Nantes': [9, 16, 14, 16], 'Napoli': [3, 1, 10, 4],
    'Newcastle United': [11, 4, 7, 6], 'Nice': [5, 9, 5, 6], 'Norwich City': [20, 22, 22, 22], 'Nottingham Forest': [22, 16, 17, 14],
    'Osasuna': [10, 7, 11, 10], 'Paris Saint-Germain': [1, 1, 1, 1], 'RB Leipzig': [4, 3, 4, 4], 'Rayo Vallecano': [12, 11, 17, 15],
    'Real Betis': [5, 6, 7, 7], 'Real Madrid': [1, 2, 1, 2], 'Real Sociedad': [6, 4, 6, 8], 'Reims': [12, 11, 9, 10],
    'Rennes': [4, 4, 10, 9], 'Roma': [6, 6, 6, 7], 'Saint-Étienne': [18, 22, 22, 16], 'Salernitana': [17, 15, 20, 22],
    'Sampdoria': [15, 20, 22, 22], 'Sassuolo': [11, 13, 19, 22], 'Schalke 04': [22, 17, 22, 22], 'Sevilla': [4, 12, 14, 12],
    'Sheffield United': [22, 22, 20, 22], 'Southampton': [15, 20, 22, 19], 'Spezia': [16, 18, 22, 22],'Strasbourg': [6, 15, 13, 13],
    'Stuttgart': [15, 16, 2, 7],'Torino': [10, 10, 9, 11], 'Tottenham Hotspur': [4, 8, 5, 5],'Toulouse': [22, 13, 11, 12],
    'Troyes': [15, 19, 22, 22],'Udinese': [12, 12, 15, 13], 'Union Berlin': [5, 4, 15, 11],'Valencia': [9, 14, 9, 13],
    'Valladolid': [22, 18, 22, 19],'Venezia': [20, 22, 22, 20], 'Villarreal': [7, 5, 8, 6],'Watford': [19, 22, 22, 22],
    'Werder Bremen': [22, 13, 9, 10],'West Ham United': [7, 14, 9, 11], 'Wolfsburg': [12, 8, 12, 12],'Wolves': [10, 13, 14, 15],
}

def get_squad_rank_map(year, full_map=squad_rank_map):
    index = year - 2021
    if (index < 0 or index >= len(full_map)):
        print("No Data For That Year")
        return

    temp_dict = {}
    for squad in full_map:
        temp_dict[squad] = full_map[squad][index]
        
    return temp_dict

def apply_squad_rank(df, year):
    df = df.copy()
    rank_map_for_year = get_squad_rank_map(year)
    
    if rank_map_for_year is None:
        return df

    df['Squad_Rank'] = df['Unnamed: 4_level_0_Squad'].map(rank_map_for_year).apply(lambda x: 21 - x if pd.notnull(x) else 0)
    df = df.drop(columns=['Unnamed: 4_level_0_Squad'])
    
    return df

In [7]:
test_2024_raw = apply_squad_rank(test_2024_raw, 2024)
train_2023_raw = apply_squad_rank(train_2023_raw, 2023)
train_2022_raw = apply_squad_rank(train_2022_raw, 2022)
train_2021_raw = apply_squad_rank(train_2021_raw, 2021)

In [8]:
train_2021_raw.head(10)

,Unnamed: 0,Unnamed: 0_x,Unnamed: 0_level_0_Rk,Name_x,Unnamed: 2_level_0_Nation,Unnamed: 3_level_0_Pos,Unnamed: 5_level_0_Age,Unnamed: 6_level_0_Born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,Unnamed: 24_level_0_Matches,Name_Clean,Unnamed: 0_y,Name_y,Value (€),Squad_Rank
0,0,0,1,Max Aarons,eng ENG,DF,21,2000,34,32,2881,32.0,0,2,2,0,0,0,8,0,0.00,0.06,0.06,0.00,0.06,Matches,max aarons,NaN,NaN,NaN,1.0
1,1,1,2,Che Adams,sct SCO,FW,25,1996,30,23,2039,22.7,7,3,10,7,0,0,0,0,0.31,0.13,0.44,0.31,0.44,Matches,che adams,NaN,NaN,NaN,6.0
2,2,2,3,Rayan Aït-Nouri,dz ALG,MF,20,2001,23,20,1828,20.3,1,2,3,1,0,0,4,0,0.05,0.10,0.15,0.05,0.15,Matches,rayan ait-nouri,673.0,Rayan Aït-Nouri,22000000.0,11.0
3,3,3,4,Kristoffer Ajer,no NOR,DF,23,1998,24,23,1995,22.2,1,3,4,1,0,0,5,0,0.05,0.14,0.18,0.05,0.18,Matches,kristoffer ajer,453.0,Kristoffer Ajer,16000000.0,8.0
4,4,4,5,Nathan Aké,nl NED,DF,26,1995,14,10,923,10.3,2,0,2,2,0,0,0,0,0.20,0.00,0.20,0.20,0.20,Matches,nathan ake,5.0,Nathan Aké,42000000.0,20.0
5,5,5,6,Marc Albrighton,eng ENG,"MF,FW",31,1989,17,11,1132,12.6,1,2,3,1,0,0,3,0,0.08,0.16,0.24,0.08,0.24,Matches,marc albrighton,NaN,NaN,NaN,13.0
6,6,6,7,Thiago Alcántara,es ESP,MF,30,1991,25,17,1534,17.0,1,4,5,1,0,0,2,0,0.06,0.23,0.29,0.06,0.29,Matches,thiago alcantara,129.0,Thiago Alcántara,15000000.0,19.0
7,7,7,8,Trent Alexander-Arnold,eng ENG,DF,22,1998,32,32,2853,31.7,2,12,14,2,0,0,2,0,0.06,0.38,0.44,0.06,0.44,Matches,trent alexander-arnold,123.0,Trent Alexander-Arnold,65000000.0,19.0
8,8,8,9,Alisson,br BRA,GK,28,1992,36,36,3240,36.0,0,1,1,0,0,0,0,0,0.00,0.03,0.03,0.00,0.03,Matches,alisson,108.0,Alisson,35000000.0,19.0
9,9,9,10,Allan,br BRA,MF,30,1991,28,25,2183,24.3,0,2,2,0,0,0,7,1,0.00,0.08,0.08,0.00,0.08,Matches,allan,501.0,Allan,6000000.0,5.0


In [9]:
def clean_dataframe(df):
    df = df.dropna(subset = ["Value (€)"])
    df = df.drop(columns = ["Name_Clean", 
                            "Unnamed: 0_y", 
                            "Name_y", 
                            "Unnamed: 0_x", 
                            "Unnamed: 0_level_0_Rk", 
                            "Unnamed: 24_level_0_Matches",
                            "Unnamed: 0"
                           ])
    df = df.rename(columns={
        'Unnamed: 3_level_0_Pos': 'Position', 
        'Unnamed: 5_level_0_Age': 'Age',
        'Unnamed: 6_level_0_Born': 'Born',
        'Unnamed: 2_level_0_Nation': 'Nationality',
        'Playing Time_MP': 'Match Played',
        'Name_x': 'Name',
        'Playing Time_Starts': 'Match Started',
        'Playing Time_Min': 'Minutes Played',
        'Playing Time_90s': 'Minutes Played / 90',
        'Performance_Gls': 'Goals',
        'Performance_Ast': 'Assists',
        'Performance_G+A': 'Goals + Assists',
        'Performance_G-PK': 'Non-Penality Goals',
        'Performance_PK': 'Penalty Kick Goals',
        'Performance_PKatt': 'Penalty Kick Attempted',
        'Performance_CrdY': 'Yellow Cards',
        'Performance_CrdR': 'Red Cards',
        'Per 90 Minutes_Gls': 'Goals Per 90 Minutes',
        'Per 90 Minutes_Ast': 'Assists Per 90 Minutes',
        'Per 90 Minutes_G+A': 'G+A Per 90 Minutes',
        'Per 90 Minutes_G-PK': 'Non-Penality Goals Per 90 Minutes',
        'Per 90 Minutes_G+A-PK': 'Non-Penalty Goals + Assists/90',
        'Value (€)': 'Value',
        # '': '',
        # '': '',
        # '': '',
        # '': '',
        # '': '',
        # '': '',
    })

    intger_column_list = ["Age", "Born", "Match Played", "Match Started", "Minutes Played", "Goals", "Assists", "Goals + Assists", 
                  "Non-Penality Goals", "Penalty Kick Goals", "Penalty Kick Attempted", "Yellow Cards", "Red Cards"]
    
    float_column_list = ["Minutes Played / 90", "Goals Per 90 Minutes", "Assists Per 90 Minutes", 
                  "G+A Per 90 Minutes", "Non-Penality Goals Per 90 Minutes", "Non-Penalty Goals + Assists/90"]

    numeric_column_list = []
    numeric_column_list.extend(intger_column_list)
    numeric_column_list.extend(float_column_list)

    for column in numeric_column_list:
        df[column] = pd.to_numeric(df[column], errors='coerce')
    
    return df


In [10]:
train_df = pd.concat([train_2023_raw, train_2022_raw, train_2021_raw], axis=0)

train_df_clean = clean_dataframe(train_df)
test_2024_clean = clean_dataframe(test_2024_raw)

In [60]:
train_df_clean["Nationality"].nunique()

102

In [67]:
train_df_clean.loc[0]

,Name,Nationality,Position,Age,Born,Match Played,Match Started,Minutes Played,Minutes Played / 90,Goals,Assists,Goals + Assists,Non-Penality Goals,Penalty Kick Goals,Penalty Kick Attempted,Yellow Cards,Red Cards,Goals Per 90 Minutes,Assists Per 90 Minutes,G+A Per 90 Minutes,Non-Penality Goals Per 90 Minutes,Non-Penalty Goals + Assists/90,Value,Squad_Rank
0,Max Aarons,eng ENG,DF,23,2000,20,13,1237,13.7,0,1,1,0,0,0,1,0,0.00,0.07,0.07,0.00,0.07,6000000.0,9.0
0,Brenden Aaronson,us USA,MF,21,2000,36,28,2372,26.4,1,3,4,1,0,0,2,0,0.04,0.11,0.15,0.04,0.15,12000000.0,2.0


In [11]:
train_df_clean.head(10)

,Name,Nationality,Position,Age,Born,Match Played,Match Started,Minutes Played,Minutes Played / 90,Goals,Assists,Goals + Assists,Non-Penality Goals,Penalty Kick Goals,Penalty Kick Attempted,Yellow Cards,Red Cards,Goals Per 90 Minutes,Assists Per 90 Minutes,G+A Per 90 Minutes,Non-Penality Goals Per 90 Minutes,Non-Penalty Goals + Assists/90,Value,Squad_Rank
0,Max Aarons,eng ENG,DF,23,2000,20,13,1237,13.7,0,1,1,0,0,0,1,0,0.00,0.07,0.07,0.00,0.07,6000000.0,9.0
2,Tyler Adams,us USA,MF,24,1999,3,1,121,1.3,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,18000000.0,9.0
3,Tosin Adarabioyo,eng ENG,DF,25,1997,20,18,1617,18.0,2,0,2,2,0,0,2,0,0.11,0.00,0.11,0.11,0.11,20000000.0,8.0
5,Simon Adingra,ci CIV,MF,21,2002,31,25,2222,24.7,6,1,7,6,0,0,3,0,0.24,0.04,0.28,0.24,0.28,28000000.0,10.0
6,Nayef Aguerd,ma MAR,DF,27,1996,21,21,1857,20.6,1,0,1,1,0,0,3,1,0.05,0.00,0.05,0.05,0.05,18000000.0,12.0
8,Naouirou Ahamada,fr FRA,MF,21,2002,20,0,349,3.9,0,0,0,0,0,0,4,1,0.00,0.00,0.00,0.00,0.00,3500000.0,11.0
10,Ola Aina,ng NGA,"DF,MF",26,1996,22,20,1692,18.8,1,1,2,1,0,0,3,0,0.05,0.05,0.11,0.05,0.11,22000000.0,4.0
11,Rayan Aït-Nouri,dz ALG,"MF,DF",22,2001,33,29,2329,25.9,2,1,3,2,0,0,7,0,0.08,0.04,0.12,0.08,0.12,35000000.0,7.0
12,Kristoffer Ajer,no NOR,DF,25,1998,28,21,1832,20.4,2,1,3,2,0,0,5,0,0.10,0.05,0.15,0.10,0.15,18000000.0,5.0
13,Manuel Akanji,ch SUI,"DF,MF",28,1995,30,28,2511,27.9,2,0,2,2,0,0,4,1,0.07,0.00,0.07,0.07,0.07,28000000.0,20.0


In [12]:
def preprocess_dataframe(df):
    df = df.copy()

    df_rest = df.drop(columns=["Value"])
    
    scaler = StandardScaler()
    numeric_cols = df_rest.select_dtypes(include=['float64', 'int64']).columns
    scaler.fit(df[numeric_cols])
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

    position_dummies = df['Position'].str.get_dummies(sep=',')
    df = pd.concat([df, position_dummies], axis=1)   
    df = df.set_index('Name')
    df = df.drop(columns = ["Nationality", "Born", "Position"])

    df = df[[c for c in df.columns if c != "Value"] + ["Value"]]
    return df, scaler

In [13]:
train_df_preprocessed, _ = preprocess_dataframe(train_df_clean)
test_2024_preprocessed, _ = preprocess_dataframe(test_2024_clean)

In [14]:
def end_to_end_process_data(df, year):
    df = apply_squad_rank(df, year)
    df = clean_dataframe(df)
    df, _ = preprocess_dataframe(df)
    return df

def end_to_end_load_data(filename):
    df = pd.read_csv(filename)
    year = int("".join(re.findall(r'\d+', filename)))

    df = end_to_end_process_data(df, year)

    return df

In [15]:
train_df_preprocessed.head(20)

,Age,Match Played,Match Started,Minutes Played,Minutes Played / 90,Goals,Assists,Goals + Assists,Non-Penality Goals,Penalty Kick Goals,Penalty Kick Attempted,Yellow Cards,Red Cards,Goals Per 90 Minutes,Assists Per 90 Minutes,G+A Per 90 Minutes,Non-Penality Goals Per 90 Minutes,Non-Penalty Goals + Assists/90,Squad_Rank,DF,FW,GK,MF,Value
Name,,,,,,,,,,,,,,,,,,,,,,,,
Max Aarons,-0.468525,-0.112396,-0.253424,-0.187787,-0.191918,-0.583455,-0.211320,-0.501001,-0.603226,-0.238984,-0.256708,-0.701791,-0.342543,-0.557663,-0.088887,-0.438495,-0.542777,-0.422959,-0.513477,1,0,0,0,6000000.0
Tyler Adams,-0.238785,-1.631729,-1.309780,-1.339169,-1.343329,-0.583455,-0.676838,-0.699107,-0.603226,-0.238984,-0.256708,-1.059343,-0.342543,-0.557663,-0.407050,-0.648993,-0.542777,-0.638929,-0.513477,0,0,0,1,18000000.0
Tosin Adarabioyo,-0.009046,-0.112396,0.186724,0.204261,0.207361,-0.021090,-0.676838,-0.302895,0.035504,-0.238984,-0.256708,-0.344239,-0.342543,-0.071922,-0.407050,-0.318210,-0.034940,-0.299547,-0.697711,1,0,0,0,20000000.0
Simon Adingra,-0.928005,0.870701,0.802931,0.828443,0.829494,1.103641,-0.211320,0.687637,1.312964,-0.238984,-0.256708,0.013313,-0.342543,0.502137,-0.225243,0.193001,0.565231,0.224952,-0.329244,0,0,0,1,28000000.0
Nayef Aguerd,0.450434,-0.023024,0.450813,0.451870,0.448786,-0.302273,-0.676838,-0.501001,-0.283861,-0.238984,-0.256708,0.013313,2.262731,-0.336872,-0.407050,-0.498637,-0.311942,-0.484665,0.039223,1,0,0,0,18000000.0
Naouirou Ahamada,-0.928005,-0.112396,-1.397809,-1.103941,-1.101904,-0.583455,-0.676838,-0.699107,-0.603226,-0.238984,-0.256708,0.370866,2.262731,-0.557663,-0.407050,-0.648993,-0.542777,-0.638929,-0.145010,0,0,0,1,3500000.0
Ola Aina,0.220694,0.066349,0.362783,0.281639,0.281646,-0.302273,-0.211320,-0.302895,-0.283861,-0.238984,-0.256708,0.013313,-0.342543,-0.336872,-0.179791,-0.318210,-0.311942,-0.299547,-1.434644,1,0,0,1,22000000.0
Rayan Aït-Nouri,-0.698265,1.049447,1.155050,0.938835,0.940921,-0.021090,-0.211320,-0.104788,0.035504,-0.238984,-0.256708,1.443522,-0.342543,-0.204397,-0.225243,-0.288138,-0.173441,-0.268695,-0.881944,1,0,0,1,35000000.0
Kristoffer Ajer,-0.009046,0.602584,0.450813,0.426078,0.430215,-0.021090,-0.211320,-0.104788,0.035504,-0.238984,-0.256708,0.728418,-0.342543,-0.116080,-0.179791,-0.197925,-0.081107,-0.176136,-1.250411,1,0,0,0,18000000.0


In [16]:
test_2024_preprocessed.head(20)

,Age,Match Played,Match Started,Minutes Played,Minutes Played / 90,Goals,Assists,Goals + Assists,Non-Penality Goals,Penalty Kick Goals,Penalty Kick Attempted,Yellow Cards,Red Cards,Goals Per 90 Minutes,Assists Per 90 Minutes,G+A Per 90 Minutes,Non-Penality Goals Per 90 Minutes,Non-Penalty Goals + Assists/90,Squad_Rank,DF,FW,GK,MF,Value
Name,,,,,,,,,,,,,,,,,,,,,,,,
Tyler Adams,-0.027815,0.546874,0.399063,0.514291,0.511219,-0.602186,0.663337,-0.143697,-0.626868,-0.235563,-0.254524,1.534666,-0.350842,-0.634828,0.428759,-0.289642,-0.627569,-0.270097,-0.354154,0,0,0,1,25000000.0
Tosin Adarabioyo,0.205776,-0.006612,-0.134254,-0.067353,-0.063146,-0.328393,-0.248159,-0.336857,-0.315504,-0.235563,-0.254524,0.408883,-0.350842,-0.344592,-0.218848,-0.326644,-0.318914,-0.309228,0.989037,1,0,0,0,20000000.0
Simon Adingra,-0.728588,0.639122,-0.400913,-0.393742,-0.392699,-0.054600,0.207589,0.049462,-0.004140,-0.235563,-0.254524,-1.092161,-0.350842,0.139134,0.590661,0.413395,0.195511,0.473399,-0.162270,0,0,0,1,22000000.0
Ola Aina,0.439367,1.192607,1.643470,1.591795,1.594037,-0.054600,-0.248159,-0.143697,-0.004140,-0.235563,-0.254524,0.784144,-0.350842,-0.344592,-0.461701,-0.474651,-0.318914,-0.465754,-0.929808,1,0,0,0,20000000.0
Rayan Aït-Nouri,-0.494997,1.377103,1.821243,1.711053,1.707026,0.492985,2.486329,1.401578,0.618589,-0.235563,-0.254524,1.534666,2.346740,-0.054356,0.914465,0.376393,-0.010259,0.434268,-1.121692,1,0,0,1,40000000.0
Kristoffer Ajer,0.205776,0.177883,0.043518,-0.037015,-0.034898,-0.602186,-0.703907,-0.723176,-0.626868,-0.235563,-0.254524,0.784144,-0.350842,-0.634828,-0.704554,-0.807669,-0.627569,-0.817936,-0.929808,1,0,0,0,16000000.0
Manuel Akanji,0.906549,0.362379,0.576836,0.565550,0.567713,-0.602186,-0.703907,-0.723176,-0.626868,-0.235563,-0.254524,0.033622,-0.350842,-0.634828,-0.704554,-0.807669,-0.627569,-0.817936,1.372806,1,0,0,1,22000000.0
Nathan Aké,0.906549,-1.113584,-0.756458,-0.832067,-0.835242,-0.602186,-0.703907,-0.723176,-0.626868,-0.235563,-0.254524,-1.092161,-0.350842,-0.634828,-0.704554,-0.807669,-0.627569,-0.817936,1.372806,1,0,0,0,18000000.0
Carlos Alcaraz,-0.962179,-1.943813,-1.467547,-1.530876,-1.532012,-0.602186,-0.703907,-0.723176,-0.626868,-0.235563,-0.254524,-1.092161,-0.350842,-0.634828,-0.704554,-0.807669,-0.627569,-0.817936,-1.889230,0,0,0,1,15000000.0


In [17]:
train_df_preprocessed.info()

<class 'pandas.DataFrame'>
Index: 5613 entries, Max Aarons to Szymon Żurkowski
Data columns (total 24 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                5613 non-null   float64
 1   Match Played                       5613 non-null   float64
 2   Match Started                      5613 non-null   float64
 3   Minutes Played                     5613 non-null   float64
 4   Minutes Played / 90                5613 non-null   float64
 5   Goals                              5613 non-null   float64
 6   Assists                            5613 non-null   float64
 7   Goals + Assists                    5613 non-null   float64
 8   Non-Penality Goals                 5613 non-null   float64
 9   Penalty Kick Goals                 5613 non-null   float64
 10  Penalty Kick Attempted             5613 non-null   float64
 11  Yellow Cards                       5613 non-null   